# Using Open Router to create Agents using different Models

In [26]:
import os
from openai import AsyncOpenAI
from agents import Agent, OpenAIChatCompletionsModel, Runner, trace
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [18]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [19]:
openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv('OPENROUTER_API_KEY')
)

def openrouter_model(model_id: str):
    return OpenAIChatCompletionsModel(
        model=model_id,
        openai_client=openrouter_client,
    )

sales_agent1 = Agent(
    name="Gemini Sales Agent",
    instructions=instructions,
    model=openrouter_model("google/gemini-3.1-flash-lite"),
)

sales_agent2 = Agent(
    name="Kimi Sales Agent",
    instructions=instructions,
    model=openrouter_model("moonshotai/kimi-k2.6"),
)

sales_agent3 = Agent(
    name="Claude Opus Sales Agent",
    instructions=instructions,
    model=openrouter_model("anthropic/claude-opus-5.5"),
)

In [20]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [21]:
tools = [tool1, tool2, tool3]

In [22]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Print the best email to the terminal
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=openrouter_model("anthropic/claude-opus-5.5"))

In [25]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

I picked **Draft 2 (sales_agent2)** as the best of the three.

---

**Subject:** SOC2 prep without the spreadsheet migraine

Hi [First Name],

Chasing screenshots, wrangling vendors, and translating your security policies into auditor-speak—SOC2 prep can feel like a second job that nobody applied for.

ComplAI automates the parts that slow teams down. Our AI continuously collects evidence, maps it to your controls, and flags gaps before your auditor ever sees them. Most customers cut their prep time by 60% and walk into audits actually confident.

Worth a brief look? I'd love to show you how [Similar Company] got audit-ready in [X weeks] without expanding their security team.

Are you open to a quick 15-minute chat next Tuesday or Wednesday?

Best,
[Your Name]
ComplAI
[Calendar Link]

**P.S.** If you've already started your SOC2 journey, I can show you how to migrate your existing work into ComplAI without starting from scratch.

---

**Why this one won:**
- **It opens on the reader's 